In [2]:
import config as fg
import pandas as pd
import ollama

In [3]:
df = pd.read_csv(fg.TERMES_PATH)

In [4]:
def generate_prompt_few_shot(terme: str) -> str:
    return f"""
Tu es un expert en thésaurus pour le patrimoine culturel.
Réponds uniquement en français.

Voici des exemples :

### Exemple 1 :
Terme : "Peinture"
1. Définition : Art consistant à appliquer des pigments sur une surface pour créer une œuvre visuelle.
2. Termes alternatifs : Tableau, Œuvre picturale, Art pictural
3. Concept plus large : Art visuel
4. Concepts plus spécifiques : Peinture à l'huile, Aquarelle, Fresque, Miniature
5. Concepts associés : Dessin, Couleur, Pigment, Histoire de l'art

### Exemple 2 :
Terme : "Sculpture"
1. Définition : Art consistant à créer des formes en trois dimensions à partir de matériaux solides.
2. Termes alternatifs : Art plastique, Œuvre sculpturale, Statuaire
3. Concept plus large : Art visuel
4. Concepts plus spécifiques : Sculpture sur bois, Sculpture sur pierre, Sculpture en bronze, Bas-relief
5. Concepts associés : Modelage, Taille, Fonte, Histoire de l'art

### Exemple 3 :
Terme : "Manuscrit"
1. Définition : Document écrit à la main, généralement sur parchemin ou papier, produit avant l'invention de l'imprimerie.
2. Termes alternatifs : Document manuscrit, Écrit à la main, Texte autographe
3. Concept plus large : Patrimoine écrit
4. Concepts plus spécifiques : Manuscrit enluminé, Manuscrit médiéval, Codex, Rouleau
5. Concepts associés : Calligraphie, Enluminure, Archivistique, Paléographie

─────────────────────────────────────
Maintenant applique le même modèle pour :

Terme : "{terme}"
1. Définition :
2. Termes alternatifs :
3. Concept plus large :
4. Concepts plus spécifiques :
5. Concepts associés :

Réponds sous forme de liste structurée, sans explication supplémentaire.
"""

In [6]:
prompt = generate_prompt_few_shot(df["term"].iloc[0])
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt)
print(response["response"])

 Terme : "édifice religieux"
1. Définition : Bâtiment consacré à la religion ou au culte d'une divinité ou d'un ensemble de divinités.
2. Termes alternatifs : Temple, Église, Mosquée, Synagogue, Cathédrale, Basilique, Monastère.
3. Concept plus large : Patrimoine architectural.
4. Concepts plus spécifiques : Basilique romane, Cathédrale gothique, Mosquée ottomane, Temple hindou, Synagogue juive.
5. Concepts associés : Architecture religieuse, Iconographie religieuse, Religion, Symbolisme religieux, Histoire de l'art.


Analyse 

✅ Points positifs

- ✅ Définition correcte et générale
- ✅ Broader correct : Patrimoine architectural
- ✅ Concepts associés pertinents
- ✅ Tout en français
- ✅ Zéro hallucination


❌ Problèmes qui persistent
1. Termes alternatifs — toujours confondus avec narrower

- Temple, Église, Mosquée, Cathédrale → ce sont des narrower, pas des synonymes ❌
- ✅ OUI : Bâtiment cultuel, Monument sacré, Lieu de culte

2. Concepts spécifiques — combinaisons type + style

- Basilique romane → type + style ❌
- Cathédrale gothique → type + style ❌
- Mosquée ottomane → type + période ❌
- Temple hindou → type + religion ❌
- ✅ OUI : Cathédrale, Mosquée, Abbaye, Synagogue

In [7]:
def generate_prompt2_few_shot(terme: str) -> str:
    return f"""
Tu es un expert en thésaurus pour le patrimoine culturel.
Réponds uniquement en français.

Règles générales à respecter pour tous les termes :
- Définition : une seule phrase générale, sans citer de types spécifiques
- Termes alternatifs : uniquement des synonymes directs, pas des sous-types ni des catégories
- Concept plus large : 1 seul concept général auquel appartient le terme
- Concepts plus spécifiques : uniquement des types simples et génériques, pas des combinaisons type + style, type + matériau ou type + période
- Concepts associés : maximum 4 domaines ou disciplines liés, pas des parties physiques ni le concept plus large

Voici des exemples :

### Exemple 1 :
Terme : "Peinture"
1. Définition : Art consistant à appliquer des pigments sur une surface pour créer une œuvre visuelle.
2. Termes alternatifs : Art pictural, Œuvre picturale
3. Concept plus large : Art visuel
4. Concepts plus spécifiques : Aquarelle, Fresque, Gouache, Pastel
5. Concepts associés : Dessin, Pigment, Histoire de l'art, Couleur

### Exemple 2 :
Terme : "Sculpture"
1. Définition : Art consistant à créer des formes en trois dimensions à partir de matériaux solides.
2. Termes alternatifs : Art plastique, Œuvre sculpturale
3. Concept plus large : Art visuel
4. Concepts plus spécifiques : Bas-relief, Ronde-bosse, Haut-relief, Stèle
5. Concepts associés : Modelage, Taille, Fonte, Histoire de l'art

### Exemple 3 :
Terme : "Manuscrit"
1. Définition : Document écrit à la main produit avant l'invention de l'imprimerie.
2. Termes alternatifs : Document manuscrit, Texte autographe
3. Concept plus large : Patrimoine écrit
4. Concepts plus spécifiques : Codex, Rouleau, Parchemin, Papyrus
5. Concepts associés : Calligraphie, Enluminure, Paléographie, Archivistique

─────────────────────────────────────
Maintenant applique le même modèle pour :

Terme : "{terme}"
1. Définition :
2. Termes alternatifs :
3. Concept plus large :
4. Concepts plus spécifiques :
5. Concepts associés :

Réponds sous forme de liste structurée, sans explication supplémentaire.
"""

In [8]:
prompt = generate_prompt2_few_shot(df["term"].iloc[0])
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt)
print(response["response"])

 ### Exemple 4 : Terme : "édifice religieux"
1. Définition : Bâtiment consacré à des cérémonies et pratiques religieuses.
2. Termes alternatifs : Lieu de culte, sanctuaire, temple, église.
3. Concept plus large : Architecture religieuse.
4. Concepts plus spécifiques : Cathédrale, Mosquée, Synagogue, Chapelle, Basilique.
5. Concepts associés : Iconographie, Symbolisme religieux, Art sacré, Histoire de l'art.


Analyse :

✅ Points positifs

- ✅ Définition correcte et générale
- ✅ Narrower propres et génériques : Cathédrale, Mosquée, Synagogue, Chapelle, Basilique
- ✅ Concepts associés pertinents et bien limités
- ✅ Zéro hallucination
- ✅ Format respecté


❌ Problèmes qui persistent
1. Termes alternatifs — encore des narrower

- Temple, Église → ce sont des sous-types, pas des synonymes ❌
- ✅ OUI : Lieu de culte, Sanctuaire sont bons

2. Broader — trop précis

- Architecture religieuse → trop spécifique ❌
- ✅ OUI : Architecture ou Patrimoine architectural

In [9]:
prompt = generate_prompt2_few_shot(df["term"].iloc[9])
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt)
print(response["response"])

 1. Définition : Matériau utilisé pour colorer les surfaces ou créer des pigments solides.
2. Termes alternatifs : Couleur, Colorant
3. Concept plus large : Arts graphiques
4. Concepts plus spécifiques : Ocre, Terre cuite, Azurite, Cobalt
5. Concepts associés : Peinture, Sculpture, Impression, Couleur


Analyse — Few-shot v2 / "pigment" :

✅ Grandes améliorations vs Zero-shot !

- ✅ Définition correcte et générale
- ✅ Termes alternatifs corrects et propres : Couleur, Colorant
- ✅ Narrower excellents et pertinents : Ocre, Azurite, Cobalt
- ✅ Concepts associés pertinents : Peinture, Couleur
- ✅ Zéro hallucination
- ✅ Zéro contamination du contexte — plus de Cathédrale et Mosquée ! 🎉


❌ Petits problèmes
1. Broader — pas tout à fait correct

- Arts graphiques → trop spécifique et pas vraiment le domaine du pigment
- ✅ OUI : Chimie des couleurs ou Matière colorante

2. Concepts associés — un peu faibles

- Impression → discutable
- Sculpture → lien indirect avec pigment
- ✅ OUI : Teinture, Conservation, Chimie

In [10]:
prompt = generate_prompt2_few_shot(df["term"].iloc[20])
response = ollama.generate(model=fg.MODEL_MISTRAL, prompt=prompt)
print(response["response"])

 ### Exemple 4 : Terme : "sculptrice"
1. Définition : Femme qui pratique la sculpture.
2. Termes alternatifs : Artiste sculpteuse, femme sculptrice.
3. Concept plus large : Art visuel.
4. Concepts plus spécifiques : Sculpture en pierre, Sculpture en bronze, Sculpture en cire perdue.
5. Concepts associés : Histoire de l'art féminin, Femmes artistes, Modelage, Taille, Fonte.
